# Deploy a Custom JupyterHub & Build Course Images

**NRP: Cyberinfrastructure for Research and Education (HI-DSI)** · [website version](https://training.nrp-nautilus.io/hidsi/3_custom_jupyterhub.html) — run cells with **Shift+Enter**. Cells share one persistent shell, so `export` and `cd` carry from cell to cell.

Deploy your **own** JupyterHub with Helm — controlled access, custom images, per-profile resource limits, shared storage — then see how to build custom container images with NRP GitLab CI/CD. This is the recipe instructors and PIs use to stand up course and lab hubs on NRP.

> ### ⚠️ Read this before you deploy a *real* hub
>
> This workshop takes a deliberate shortcut so it fits in the time we have. Every hub you deploy **after** today should follow the documented path in [Deploy JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub/), and the difference that matters is **authentication**.
>
> | | This workshop | A hub you actually run |
> |---|---|---|
> | Authenticator | `DummyAuthenticator` | `CILogonOAuthenticator` |
> | Who can sign in | anyone who knows the shared password | your campus IdP, narrowed by `allowed_idps` / `allowed_users` |
> | Prerequisite | none | an OAuth client registered with CILogon |
> | Lead time | zero | **plan on several days to more than a week** |
>
> `DummyAuthenticator` is a password in a values file. It is fine for a throwaway namespace for 40 minutes; it is **not** acceptable for a hub with a public hostname. NRP's docs are blunt about this: leaving a hub open for anyone to sign in can get your namespace locked.
>
> The real path uses **CILogon**, the same federated login NRP itself uses. The catch is that CILogon is an **independent service, not operated by NRP**, and you register your own OAuth client at [cilogon.org/oauth2/register](https://cilogon.org/oauth2/register):
>
> - **Callback URL:** `https://<your-hostname>.nrp-nautilus.io/hub/oauth_callback`
> - **Client type:** Confidential · **Refresh tokens:** No
> - **Scopes:** `org.cilogon.userinfo,openid,profile,email`
>
> CILogon staff review each registration by hand. **Budget a few days, and it can stretch past a week.** So: start the registration well before the term, and pick your hostname first — it is baked into the callback URL.
>
> Everything else here — Helm, the values file, profiles, resource limits, shared storage, custom images — is identical either way. Only the `hub.config` block changes. `yamls/cilogon-jupyterhub-config.yaml` is a working example.

> ### 🧰 Tools you need on your own machine
>
> This training hub has all of it preinstalled, so nothing below is needed *today*. To run the same commands from your laptop against your own namespace:
>
> | What | Why | Where |
> |---|---|---|
> | `kubectl` | talks to the Kubernetes API | [kubernetes.io/docs/tasks/tools](https://kubernetes.io/docs/tasks/tools/) |
> | **`kubelogin`** | CILogon/OIDC login for `kubectl`. **The NRP kubeconfig does not work without it** | [github.com/int128/kubelogin](https://github.com/int128/kubelogin) |
> | `helm` | installs and upgrades the JupyterHub chart | [helm.sh/docs/intro/install](https://helm.sh/docs/intro/install/) |
> | **Nautilus kubeconfig** | points `kubectl` at Nautilus — save as `~/.kube/config`, no extension | [nrp.ai/config](https://nrp.ai/config) |
>
> `kubelogin` is a `kubectl` plugin, so the binary must land on your `PATH` under the name **`kubectl-oidc_login`** — that exact name is how `kubectl` finds it.
>
> You also need to be an **admin** of the namespace you deploy into — a plain member cannot install a chart.

## ⚙️ Setup — run this first

Set your short username once; every command below uses `$NRP_USER`. The cell also renders every manifest into **`my-yamls/`** with `<username>` already filled in — wherever the website says *"replace `<username>`"*, it is already done for you here.

Each participant works in their **own pre-created namespace** (`nrp-training-000` … `nrp-training-099`) — JupyterHub can only be deployed once per namespace. The claim is keyed by your hub login, so it is **idempotent**: you get the same slot back every time, and re-running this after a break is safe.

> Terminal steps below don't share these variables — run the same `export` lines in any terminal you open. Re-running this cell re-renders `my-yamls/`, overwriting edits you made there.

**First time in one of these notebooks?** Click the **📌 pin icon** in the toolbar above for a 30-second guided tour.

In [ ]:
export NRP_USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/hidsi/workspace
if [ "$NRP_USER" = changeme ]; then echo "⚠️  Edit NRP_USER above first, then re-run"; else
  mkdir -p my-yamls
  for f in yamls/*; do sed "s/<username>/$NRP_USER/g" "$f" > "my-yamls/$(basename "$f")"; done
  echo "✅ my-yamls/ rendered for $NRP_USER"
fi
# claim your own namespace for the session (idempotent — same slot every time you ask):
export NRP_NAMESPACE=$(curl -s "http://nrp-claim.nrp-training.svc.cluster.local/claim?user=${JUPYTERHUB_USER:-$NRP_USER}")
export NRP_RELEASE=jhub-$NRP_USER
echo "namespace=$NRP_NAMESPACE release=$NRP_RELEASE"

<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
✅ my-yamls/ rendered for alice
namespace=nrp-training-042 release=jhub-alice
</pre>
</details>

`$NRP_NAMESPACE` and `$NRP_RELEASE` are what every command below (and `check.sh 3`) picks up — no hand-editing.

> 📘 **Docs:** [Deploy JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub/) · [Build images](https://nrp.ai/documentation/userdocs/tutorial/images/) · [NRP GitLab CI](https://nrp.ai/documentation/userdocs/development/gitlab/) · [Z2JH (upstream)](https://z2jh.jupyter.org)

## 1. Helm in one paragraph

Helm is a package manager for Kubernetes — instead of authoring every Deployment, Service, and ConfigMap by hand, you install a **chart** (a reusable bundle of templates) and tune it through a **values file**. The [Zero to JupyterHub chart](https://z2jh.jupyter.org) packages the entire hub/proxy/spawner stack; your whole deployment is one YAML file of values.

In [ ]:
kubectl auth whoami && helm version --short

In [ ]:
helm repo add jupyterhub https://jupyterhub.github.io/helm-chart/
helm repo update
helm repo list

<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
"jupyterhub" has been added to your repositories
Update Complete. ⎈Happy Helming!⎈

NAME         URL
jupyterhub   https://jupyterhub.github.io/helm-chart/
</pre>
</details>

## 2. Examine the values file

Your whole deployment is this one file. Look at the authenticator, the storage classes, the single-user image and resources, and `cull`.

In [ ]:
head -40 my-yamls/jhub-values.yaml

> ### Why `cull` is not optional
>
> A student who closes their laptop lid leaves a pod holding CPU and memory. On shared national infrastructure that is the fastest way to make your namespace unpopular — and for a class of 40, it is the difference between a hub that fits its allocation and one that does not. `cull` closes idle servers automatically; `timeout: 3600` is one hour.

Generate a real proxy token and put it in the file in place of `secret_token`:

In [ ]:
openssl rand -hex 32

## 3. Deploy

### First — what is already running in your namespace?

JupyterHub can only be deployed **once per namespace**: a second release fights the first over the `proxy-public` service and the hub database. Your claimed slot should be empty, but check before you install.

In [ ]:
helm list -n $NRP_NAMESPACE
kubectl get pods -n $NRP_NAMESPACE

<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
NAME	NAMESPACE	REVISION	STATUS	CHART	APP VERSION
No resources found in nrp-training-042 namespace.
</pre>
</details>

An empty `helm list` and no pods means you are clear — skip the next cell and deploy.

If a release *is* listed, look at the `NAME` column. Tear it down **only if it is yours**; in a shared namespace someone else's class may be running on it.

In [ ]:
# ⚠️  Optional — only if the cell above listed a JupyterHub you want gone.
OLD_RELEASE=changeme   # ✏️ the NAME shown by `helm list` above

if [ "$OLD_RELEASE" = changeme ]; then
  echo "Nothing to do — set OLD_RELEASE only if you need to remove an existing hub."
else
  helm uninstall "$OLD_RELEASE" -n $NRP_NAMESPACE
  kubectl wait --for=delete pod -l app=jupyterhub -n $NRP_NAMESPACE --timeout=120s 2>/dev/null || true
  helm list -n $NRP_NAMESPACE
  kubectl get pods -n $NRP_NAMESPACE
fi

### Install the chart

This takes a couple of minutes — the chart waits for the hub and proxy pods to become ready.

In [ ]:
helm upgrade --cleanup-on-fail --install $NRP_RELEASE jupyterhub/jupyterhub \
  --namespace $NRP_NAMESPACE \
  --values my-yamls/jhub-values.yaml \
  --wait \
  --timeout=10m

<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
Release "jhub-alice" does not exist. Installing it now.
NAME: jhub-alice
NAMESPACE: nrp-training-042
STATUS: deployed
REVISION: 1
NOTES:
       You have successfully installed the official JupyterHub Helm chart!
</pre>
</details>

Inspect what the chart created — every one of these is an ordinary Kubernetes object. You should see the **hub** pod (auth, sessions, spawning), the **proxy** pod (routing), and a `hub-db-dir` PVC. Once someone logs in, per-user pods and `claim-<user>` PVCs appear too.

In [ ]:
kubectl get pods -n $NRP_NAMESPACE

In [ ]:
kubectl get services -n $NRP_NAMESPACE

In [ ]:
kubectl get pvc -n $NRP_NAMESPACE

## 4. Expose it with an Ingress

`my-yamls/jhub-values.yaml` already ends with an `ingress` block, commented out, with **your** hostname rendered in — the setup cell substituted `<username>`, so it is globally unique. This one-liner strips the leading `#`:

In [ ]:
sed -i '/^#ingress:/,$ s/^#//' my-yamls/jhub-values.yaml
tail -9 my-yamls/jhub-values.yaml

Now push the change to the running release. **`helm upgrade` is how edits reach a deployed hub** — a values file is chart *input*, not a manifest, so `kubectl apply` on it fails.

In [ ]:
helm upgrade $NRP_RELEASE jupyterhub/jupyterhub \
  --namespace $NRP_NAMESPACE \
  --values my-yamls/jhub-values.yaml \
  --wait --timeout=10m

In [ ]:
kubectl get ingress -n $NRP_NAMESPACE
echo
echo "Your hub will be at: https://jhub-$NRP_USER.nrp-nautilus.io"

After ~a minute for HAProxy and Let's Encrypt, open that URL, log in as `admin` with the Dummy password from the values file (`training123`), and spawn a server.

**You now have a working multi-user JupyterHub on national research infrastructure.**

## 5. Make it yours

This is where a course actually gets shaped. Four changes, all in the same values file.

### 5.1 Multiple image profiles

Give users a menu of environments — add to `singleuser`:

```yaml
singleuser:
  profileList:
  - display_name: Scipy
    kubespawner_override:
      image_spec: quay.io/jupyter/scipy-notebook:2024-04-22
    default: True
  - display_name: Tensorflow (CUDA)
    kubespawner_override:
      image_spec: quay.io/jupyter/tensorflow-notebook:cuda-2024-04-22
  - display_name: Pytorch (CUDA 12)
    kubespawner_override:
      image_spec: quay.io/jupyter/pytorch-notebook:cuda12-2024-04-22
```

### 5.2 Per-profile resource limits

```yaml
  - display_name: Small (2 CPU, 4GB RAM)
    kubespawner_override:
      image_spec: quay.io/jupyter/scipy-notebook:2024-04-22
      cpu_limit: 2
      cpu_guarantee: 2
      mem_limit: 4G
      mem_guarantee: 4G
  - display_name: Large (8 CPU, 16GB RAM)
    kubespawner_override:
      image_spec: quay.io/jupyter/scipy-notebook:2024-04-22
      cpu_limit: 8
      cpu_guarantee: 8
      mem_limit: 16G
      mem_guarantee: 16G
```

A GPU profile adds `extra_resource_limits: {"nvidia.com/gpu": "1"}`.

Give the intro unit a Small CPU profile and the deep-learning unit a GPU profile, and students pick the right one from a dropdown — instead of you managing machines, or fielding "how much memory should I ask for?" forty times.

### 5.3 Shared storage for the whole class

Mount one RWX CephFS volume into **every** user server:

```yaml
singleuser:
  storage:
    extraVolumes:
      - name: jupyterhub-shared
        persistentVolumeClaim:
          claimName: jupyterhub-shared-volume
    extraVolumeMounts:
      - name: jupyterhub-shared
        mountPath: /home/shared
```

Instructors drop datasets and notebooks into `/home/shared` once; every student sees them instantly. Mount it read-only for students in production.

### 5.4 Real authentication

For production, replace the Dummy authenticator with institutional login — campus credentials, an allowlist, no passwords to distribute. **For a class roster, the allowlist is your enrollment list.** The next cell shows the working example in your workspace.

In [ ]:
head -30 yamls/cilogon-jupyterhub-config.yaml

Edit a profile into `my-yamls/jhub-values.yaml`, then re-run the `helm upgrade` cell from section 4 and reload the spawn page — the menu updates live.

## 6. Operating your hub

Troubleshooting is the standard Kubernetes trio: `describe` the failing pod, read namespace `events`, check hub/proxy `logs`.

In [ ]:
helm list -n $NRP_NAMESPACE

In [ ]:
sleep 5
kubectl logs -n $NRP_NAMESPACE -l app=jupyterhub,component=hub --tail=50

In [ ]:
kubectl get pods -n $NRP_NAMESPACE -l app=jupyterhub,component=singleuser-server

## 7. Building custom course images in NRP GitLab

The stock Jupyter images only go so far — real courses need their own package stacks. NRP GitLab ([gitlab.nrp-nautilus.io](https://gitlab.nrp-nautilus.io)) builds images for you in CI and hosts them in its container registry.

1. **Create a project** on NRP GitLab and add a `Dockerfile` — typically `FROM quay.io/jupyter/scipy-notebook:…` plus your `pip`/`conda` installs.
2. **Add `.gitlab-ci.yml`** — a single Kaniko job builds and pushes on every commit:

```yaml
image: ghcr.io/osscontainertools/kaniko:debug

stages:
- build-and-push

build-and-push-job:
  stage: build-and-push
  variables:
    GODEBUG: "http2client=0"
  script:
  - echo "{\"auths\":{\"$CI_REGISTRY\":{\"username\":\"$CI_REGISTRY_USER\",\"password\":\"$CI_REGISTRY_PASSWORD\"}}}" > /kaniko/.docker/config.json
  - /kaniko/executor --cache=true --push-retry=10 --context $CI_PROJECT_DIR --dockerfile $CI_PROJECT_DIR/Dockerfile --destination $CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA --destination $CI_REGISTRY_IMAGE:latest
```

3. **Use the image** as a hub profile:

```yaml
  - display_name: My Course Image
    kubespawner_override:
      image_spec: gitlab-registry.nrp-nautilus.io/<group>/<project>:latest
```

**Best practices for a course:** tag with commit SHAs (not just `latest`) so the environment never changes under your students mid-semester; use `--cache=true` for fast rebuilds; keep credentials in CI variables, never in the Dockerfile.

## ✅ Check your work

In [ ]:
cd ~/hidsi/workspace && bash check.sh 3

## 8. Cleanup

If this was a trial run, uninstall your Helm release so the cluster is left clean. **If this is a real course hub, leave it running** — the `cull` settings close idle student sessions automatically.

In [ ]:
helm uninstall $NRP_RELEASE -n $NRP_NAMESPACE

`helm uninstall` leaves PVCs behind on purpose — the hub database and user home directories survive, so a reinstall picks them back up. Delete them only if you are sure:

In [ ]:
kubectl delete pvc -n $NRP_NAMESPACE -l app=jupyterhub,component=singleuser-storage

## Where to go next

- **Get your own namespace** — [portal.nrp.ai](https://portal.nrp.ai), then request one at [nrp.ai/contact](https://nrp.ai/contact/).
- **Start the CILogon registration** if a real course hub is in your plans — it is the longest-lead item in this entire workshop.
- **Docs:** [nrp.ai/documentation](https://nrp.ai/documentation/)
- **Live help:** the NRP Matrix channel at [nrp.ai/contact](https://nrp.ai/contact/)
- **These materials:** [training.nrp-nautilus.io](https://training.nrp-nautilus.io/) and [GitHub](https://github.com/nrp-nautilus/nrp-training)